# OpenBCI Ganglion 신경신호 실습

**재활재생개론 · 신경신호 수집과 처리**

---

## 이 실습에서 하는 일

| 단계 | 내용 |
| --- | --- |
| 1 | BLED112 동글로 Ganglion 보드에 연결 |
| 2 | 신경신호(EEG/EMG) 실시간 수집 |
| 3 | 필터링 — 잡음 제거 |
| 4 | RMS · 주파수 분석 |
| 5 | 그래프로 확인하고 CSV로 저장 |

## 준비물

- **OpenBCI Ganglion 보드** (전원 ON)
- **BLED112 USB 동글** (COM 포트에 연결)
- 보드의 이름 또는 MAC 주소 (예: `Ganglion-3587`)

## 중요한 준비 단계

> ⚠️ **OpenBCI GUI를 반드시 종료하세요.**
> GUI가 동글을 점유하고 있으면 파이썬이 연결할 수 없습니다.
> 하나의 동글은 한 번에 하나의 프로그램만 사용할 수 있습니다.

## Ganglion 보드 사양

| 항목 | 값 |
| --- | --- |
| EEG 채널 수 | **4개** |
| 샘플링 레이트 | **200 Hz** |
| 가속도계 | 3축 |
| 통신 | BLE (BLED112 동글 경유) |

> 💡 Cyton 보드(8채널·250Hz)와 사양이 다릅니다. 이 노트북은 **Ganglion 전용**입니다.

---

**위에서 아래로 셀을 하나씩 실행하세요.** (`Shift + Enter`)

---
# 1단계 · 라이브러리 준비

필요한 도구를 불러옵니다.

**BrainFlow**는 OpenBCI 공식 라이브러리로, OpenBCI GUI도 내부적으로 이것을 사용합니다.
우리가 직접 통신 프로토콜을 구현할 필요 없이 보드와 대화할 수 있게 해 줍니다.

> 처음 실행한다면 아래 셀이 자동으로 `brainflow`를 설치합니다 (1~2분).

In [ ]:
import sys
import subprocess

# 콘솔에서 한글과 µ 기호가 깨지지 않도록 (Windows cp949 대응)
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

# brainflow 가 없으면 자동 설치
try:
    import brainflow
except ImportError:
    print('brainflow 설치 중... (1~2분 소요)')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'brainflow'])
    import brainflow

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from scipy import signal as sp_signal

from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds

# 그래프 한글 표시
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# brainflow 모듈에는 __version__ 속성이 없어서 패키지 메타데이터에서 조회한다
from importlib.metadata import version as _pkg_version
try:
    bf_version = _pkg_version('brainflow')
except Exception:
    bf_version = '(확인 불가)'

print('준비 완료')
print(f'  Python    : {sys.version.split()[0]}')
print(f'  BrainFlow : {bf_version}')
print(f'  NumPy     : {np.__version__}')
print(f'  SciPy     : {__import__("scipy").__version__}')

---
# 2단계 · 동글이 꽂힌 COM 포트 찾기

BLED112 동글은 컴퓨터에 **가상 시리얼 포트**(COM3, COM4 …)로 나타납니다.
OpenBCI GUI에서 선택했던 것과 **같은 포트 번호**를 쓰면 됩니다.

아래 셀이 후보를 찾아 줍니다. 보통 이름에 `Bluetooth` 또는 `CP210x`가 들어갑니다.

In [ ]:
try:
    from serial.tools import list_ports
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyserial'])
    from serial.tools import list_ports

ports = list(list_ports.comports())

print('연결된 COM 포트')
print('=' * 60)

if not ports:
    print('포트를 찾지 못했습니다.')
    print('  -> BLED112 동글이 USB 에 꽂혀 있는지 확인하세요.')
else:
    for p in ports:
        # 동글일 가능성이 높은 포트에 표시
        desc = (p.description or '').lower()
        hint = '  <-- 동글일 가능성 높음' if ('bluetooth' in desc or 'cp210' in desc) else ''
        print(f'  {p.device:<8} {p.description}{hint}')

print('=' * 60)
print('위 목록에서 동글의 포트 번호를 확인하고, 다음 셀에 적으세요.')

---
# 3단계 · 설정 ⭐ **여기만 고치면 됩니다**

이 노트북에서 **학생이 직접 수정하는 곳은 이 셀 하나뿐**입니다.

### `COM_PORT`
2단계에서 확인한 동글 포트. OpenBCI GUI에서 쓰던 것과 같은 번호.

### `BOARD_MAC` — 어떤 Ganglion 보드에 연결할지 지정

| 상황 | 설정값 |
| --- | --- |
| 주변에 Ganglion이 **내 것 하나뿐** | `''` (빈 문자열) → 자동으로 찾음 |
| **강의실처럼 여러 보드**가 켜져 있음 | 내 보드의 MAC 주소를 입력 ← **필수** |

> ⚠️ 여러 보드가 동시에 켜져 있는데 비워 두면 **옆 사람 보드에 연결될 수 있습니다.**
> 강의실에서는 반드시 MAC 주소를 지정하세요.

MAC 주소를 모른다면 → **4단계**에서 찾는 방법을 안내합니다. 일단 `''`로 두고 진행해도 됩니다.

### `DURATION_SEC`
몇 초 동안 신호를 모을지. 처음에는 10초를 권장합니다.

In [ ]:
# ===================== 학생 설정 구역 =====================

COM_PORT     = 'COM3'      # 동글 포트

BOARD_MAC    = ''          # 예: 'd2:b4:11:81:48:ad'
                           # 비워두면 자동 탐색 (보드가 하나일 때만 안전)

DURATION_SEC = 10          # 수집 시간 (초)

# ==========================================================

BOARD_ID = BoardIds.GANGLION_BOARD   # BLED112 동글용 Ganglion (board_id = 1)

print('설정 확인')
print('=' * 60)
print(f'  동글 포트   : {COM_PORT}')
print(f'  대상 보드   : {BOARD_MAC if BOARD_MAC else "(자동 탐색)"}')
print(f'  수집 시간   : {DURATION_SEC} 초')
print('=' * 60)

if not BOARD_MAC:
    print('주의: 자동 탐색 모드입니다.')
    print('      주변에 다른 Ganglion 이 켜져 있으면 그쪽에 연결될 수 있습니다.')

---
# 4단계 · 보드 사양 확인 (연결 없이)

BrainFlow는 보드에 연결하지 않아도 사양을 알려줍니다.
**내가 다루는 신호가 몇 채널이고 초당 몇 개인지** 먼저 확인하는 습관을 들이세요.

이 값들은 뒤의 모든 분석(필터 설계, 주파수 축)의 기준이 됩니다.

In [ ]:
FS           = BoardShim.get_sampling_rate(BOARD_ID)   # 샘플링 레이트 (Hz)
EEG_CHANNELS = BoardShim.get_eeg_channels(BOARD_ID)    # EEG 데이터가 담긴 행 번호
N_EEG        = len(EEG_CHANNELS)

print('Ganglion 보드 사양')
print('=' * 60)
print(f'  board_id         : {int(BOARD_ID)}')
print(f'  샘플링 레이트    : {FS} Hz')
print(f'  EEG 채널 수      : {N_EEG} 개')
print(f'  EEG 행 인덱스    : {EEG_CHANNELS}')
print(f'  전체 행 수       : {BoardShim.get_num_rows(BOARD_ID)}')
print('=' * 60)
print()
print(f'{DURATION_SEC}초 수집하면 채널당 약 {FS * DURATION_SEC}개의 샘플이 모입니다.')
print()
print('참고: BrainFlow 는 데이터를 (행=채널, 열=시간) 행렬로 줍니다.')
print('      EEG 행 인덱스는 그 행렬에서 EEG 가 들어있는 줄 번호입니다.')

### 💡 MAC 주소를 모를 때

OpenBCI GUI에 보이는 `Ganglion-3587` 은 **블루투스 광고 이름**이고,
BrainFlow의 `mac_address`는 **MAC 주소**(`d2:b4:11:81:48:ad` 형태)를 받습니다. 서로 다릅니다.

**MAC 주소 찾는 법 (Windows)**

1. Microsoft Store에서 **Bluetooth LE Explorer** 설치
2. 앱 실행 → `Start` 로 스캔
3. 목록에서 `Ganglion-3587` 을 찾으면 옆에 MAC 주소가 표시됨
4. 그 값을 3단계 `BOARD_MAC` 에 붙여넣기

**보드별 MAC을 한 번 정리해 두면** 이후 수업에서 계속 재사용할 수 있습니다.

| 보드 이름 | MAC 주소 |
| --- | --- |
| Ganglion-3587 | (여기에 기록) |
| Ganglion-xxxx | (여기에 기록) |

---
# 5단계 · 보드 연결

`BrainFlowInputParams` 에 접속 정보를 담아 보드를 엽니다.

- `serial_port` → 동글이 꽂힌 COM 포트 (**필수**)
- `mac_address` → 연결할 보드 지정 (**선택**, 비우면 자동 탐색)
- `timeout` → 몇 초까지 기다릴지

`prepare_session()` 이 성공하면 연결된 것입니다. 보드 LED가 바뀌는 것도 확인해 보세요.

In [ ]:
BoardShim.disable_board_logger()   # 로그를 보려면 enable_dev_board_logger() 로 변경

params = BrainFlowInputParams()
params.serial_port = COM_PORT
params.mac_address = BOARD_MAC     # 빈 문자열이면 자동 탐색
params.timeout     = 20            # 초

board = BoardShim(BOARD_ID, params)

print(f'{COM_PORT} 를 통해 Ganglion 연결 시도 중...')
print('보드 전원이 켜져 있는지 확인하세요. 최대 20초 소요됩니다.')
print()

try:
    board.prepare_session()
    connected = True
    print('연결 성공')
    print(f'  대상: {BOARD_MAC if BOARD_MAC else "자동 탐색된 보드"}')
except Exception as e:
    connected = False
    print('연결 실패')
    print(f'  {e}')
    print()
    print('점검 목록')
    print('  1. OpenBCI GUI 를 종료했나요?  (가장 흔한 원인)')
    print('  2. 보드 전원 스위치가 ON 인가요?')
    print('  3. COM_PORT 번호가 맞나요?  (2단계 결과와 대조)')
    print('  4. 배터리가 충분한가요?')
    print('  5. MAC 주소를 넣었다면, 형식이 맞나요?  (비워서 자동탐색으로 먼저 시도해 보세요)')

---
# 6단계 · 신호 수집

`start_stream()` 을 부르면 보드가 데이터를 보내기 시작하고,
BrainFlow가 내부 버퍼에 계속 쌓아 둡니다.

설정한 시간만큼 기다린 뒤 `get_board_data()` 로 한 번에 가져옵니다.

> 🧪 **해 볼 것**: 수집되는 동안 눈을 깜빡이거나 이를 꽉 물어 보세요.
> 나중에 그래프에서 그 순간이 큰 파형으로 보입니다.

In [ ]:
if not connected:
    print('보드가 연결되지 않았습니다. 5단계를 먼저 성공시키세요.')
else:
    board.start_stream()
    print(f'{DURATION_SEC}초 동안 수집합니다.')
    print()

    for sec in range(DURATION_SEC):
        time.sleep(1)
        bar = '#' * (sec + 1) + '.' * (DURATION_SEC - sec - 1)
        print(f'  [{bar}] {sec + 1}/{DURATION_SEC} 초', end='\r')

    data = board.get_board_data()   # (행=채널, 열=시간)
    board.stop_stream()

    n_samples = data.shape[1]
    print()
    print()
    print('수집 완료')
    print(f'  데이터 형태 : {data.shape}  (행=채널, 열=시간)')
    print(f'  샘플 수     : {n_samples} 개')
    print(f'  실제 시간   : {n_samples / FS:.2f} 초')

    if n_samples == 0:
        print()
        print('샘플이 0개입니다. 전극이 연결되어 있는지, 보드가 켜져 있는지 확인하세요.')

---
# 7단계 · 표로 정리

BrainFlow가 준 행렬에서 **EEG 채널만 뽑아** 표(DataFrame)로 바꿉니다.
이 형태가 되면 Excel처럼 다루기 쉽고, 나중에 CSV로 저장하기도 편합니다.

단위는 **µV(마이크로볼트)** 입니다.

In [ ]:
eeg = data[EEG_CHANNELS, :]                    # EEG 행만 추출 -> (4, N)
t   = np.arange(eeg.shape[1]) / FS             # 시간축 (초)

df = pd.DataFrame(eeg.T, columns=[f'Ch{i+1}' for i in range(N_EEG)])
df.insert(0, 'time_sec', t)

print(f'표 크기: {df.shape[0]} 행 x {df.shape[1]} 열')
print()
print('처음 5개 샘플:')
display(df.head())

print()
print('채널별 기초 통계 (µV):')
display(df[[f'Ch{i+1}' for i in range(N_EEG)]].describe().loc[['mean', 'std', 'min', 'max']].round(2))

---
# 8단계 · 원본 신호 그려 보기

먼저 **아무 처리도 하지 않은 날것의 신호**를 봅니다.

대개 이렇게 보입니다.
- 신호 전체가 0에서 멀리 떨어져 위아래로 떠 있음 → **DC 오프셋**
- 천천히 흐르는 큰 물결 → **드리프트** (전극 접촉 변화, 움직임)
- 촘촘한 잔물결 → **60Hz 전원 잡음** 등

이것이 바로 다음 단계에서 필터가 필요한 이유입니다.

In [ ]:
fig, axes = plt.subplots(N_EEG, 1, figsize=(13, 2.2 * N_EEG), sharex=True)
fig.suptitle('원본 신호 (필터 전)', fontsize=15, fontweight='bold')

for i in range(N_EEG):
    axes[i].plot(t, eeg[i], linewidth=0.6, color='#888888')
    axes[i].set_ylabel(f'Ch{i+1}\n(µV)')
    axes[i].grid(alpha=0.3)

axes[-1].set_xlabel('시간 (초)')
plt.tight_layout()
plt.show()

---
# 9단계 · 필터링

관심 있는 주파수만 남기고 나머지를 걷어냅니다.

**대역통과 필터 (Bandpass)**
- 낮은 쪽 차단 → DC 오프셋과 느린 드리프트 제거
- 높은 쪽 차단 → 고주파 잡음 제거

**노치 필터 (Notch)**
- 한국 전원 주파수인 **60Hz**만 콕 집어 제거

> 🧪 **해 볼 것**: `LOW`, `HIGH` 값을 바꿔 가며 재실행해 보세요.
> 대역을 좁히면 무엇이 사라지고 무엇이 남는지 관찰하는 것이 이 실습의 핵심입니다.
>
> - EEG 뇌파 관찰 → `LOW=1, HIGH=45`
> - EMG 근전도 관찰 → `LOW=20, HIGH=95`

In [ ]:
# ---- 필터 설정 (바꿔가며 실험해 보세요) ----
LOW   = 1.0     # Hz, 이보다 낮은 성분 제거
HIGH  = 45.0    # Hz, 이보다 높은 성분 제거
NOTCH = 60.0    # Hz, 전원 잡음 (한국 60Hz / 유럽 50Hz)
ORDER = 4
# -------------------------------------------

nyq = FS / 2    # 나이퀴스트 주파수 = 표현 가능한 최대 주파수

b_bp, a_bp = sp_signal.butter(ORDER, [LOW / nyq, HIGH / nyq], btype='band')
b_no, a_no = sp_signal.iirnotch(NOTCH / nyq, Q=30)

filtered = np.zeros_like(eeg)
for i in range(N_EEG):
    x = sp_signal.filtfilt(b_bp, a_bp, eeg[i])   # 대역통과
    x = sp_signal.filtfilt(b_no, a_no, x)        # 노치
    filtered[i] = x

print(f'필터 적용 완료 : {LOW}-{HIGH} Hz 대역통과 + {NOTCH} Hz 노치')
print(f'나이퀴스트 주파수 : {nyq} Hz  (이보다 높은 주파수는 볼 수 없음)')
print()
print('채널별 변화 (표준편차, µV)')
print('-' * 46)
print(f'{"채널":<8}{"필터 전":>14}{"필터 후":>14}')
print('-' * 46)
for i in range(N_EEG):
    print(f'Ch{i+1:<7}{eeg[i].std():>14.2f}{filtered[i].std():>14.2f}')
print('-' * 46)

---
# 10단계 · 필터 전후 비교

같은 신호를 겹쳐 그려 필터가 무엇을 걷어냈는지 눈으로 확인합니다.

회색이 원본, 파란색이 필터 후입니다.

In [ ]:
fig, axes = plt.subplots(N_EEG, 1, figsize=(13, 2.2 * N_EEG), sharex=True)
fig.suptitle(f'필터 전후 비교  ({LOW}-{HIGH} Hz)', fontsize=15, fontweight='bold')

for i in range(N_EEG):
    axes[i].plot(t, eeg[i] - eeg[i].mean(), linewidth=0.5,
                 color='#cccccc', label='원본')
    axes[i].plot(t, filtered[i], linewidth=0.8,
                 color='#1f77b4', label='필터 후')
    axes[i].set_ylabel(f'Ch{i+1}\n(µV)')
    axes[i].grid(alpha=0.3)
    if i == 0:
        axes[i].legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('시간 (초)')
plt.tight_layout()
plt.show()

---
# 11단계 · RMS — 신호의 세기

**RMS(제곱평균제곱근)** 는 신호가 얼마나 강한지를 하나의 숫자로 나타냅니다.

$$\text{RMS} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} x_i^2}$$

**왜 중요한가**: 근육에 힘을 주면 EMG의 RMS가 확 올라갑니다.
그래서 *RMS가 기준값을 넘으면 모터를 움직인다* 같은 제어가 가능합니다.

MATLAB 실습에서 쓰던 `rms(s) > threshold` 방식이 바로 이것입니다.

아래는 **짧은 구간마다 RMS를 계산**해서 시간에 따라 세기가 어떻게 변하는지 봅니다.

In [ ]:
WINDOW_SEC = 0.25                      # RMS 계산 구간 길이 (초)
win        = int(FS * WINDOW_SEC)
n_win      = filtered.shape[1] // win

rms      = np.zeros((N_EEG, n_win))
rms_time = (np.arange(n_win) + 0.5) * WINDOW_SEC

for i in range(N_EEG):
    for w in range(n_win):
        seg = filtered[i, w * win:(w + 1) * win]
        rms[i, w] = np.sqrt(np.mean(seg ** 2))

THRESHOLD = 15.0   # µV, 활성 판정 기준 (신호를 보고 조정하세요)

fig, ax = plt.subplots(figsize=(13, 4.5))
for i in range(N_EEG):
    ax.plot(rms_time, rms[i], marker='o', markersize=3, label=f'Ch{i+1}')

ax.axhline(THRESHOLD, color='red', linestyle='--', linewidth=1.2,
           label=f'기준선 {THRESHOLD} µV')
ax.set_xlabel('시간 (초)')
ax.set_ylabel('RMS (µV)')
ax.set_title(f'{WINDOW_SEC}초 구간별 신호 세기', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('채널별 평균 RMS')
print('-' * 40)
for i in range(N_EEG):
    over = int(np.sum(rms[i] > THRESHOLD))
    print(f'  Ch{i+1} : {rms[i].mean():7.2f} µV   기준선 초과 {over}/{n_win} 구간')

---
# 12단계 · 주파수 분석

지금까지는 *시간에 따라* 신호를 봤습니다.
이번에는 **어떤 주파수 성분이 얼마나 들어있는지** 봅니다. (Welch 방법)

### 뇌파의 주요 대역

| 대역 | 주파수 | 관련된 상태 |
| --- | --- | --- |
| Delta | 0.5–4 Hz | 깊은 수면 |
| Theta | 4–8 Hz | 졸음, 명상 |
| **Alpha** | **8–12 Hz** | **눈 감고 이완** |
| Beta | 12–30 Hz | 각성, 집중 |
| Gamma | 30–45 Hz | 고차 인지 |

> 🧪 **고전적인 실험**: 눈을 감고 20초 수집 → 눈을 뜨고 20초 수집.
> 눈을 감았을 때 **Alpha가 뚜렷하게 커집니다.** 직접 확인해 보세요.

In [ ]:
BANDS = {
    'Delta': (0.5, 4),
    'Theta': (4, 8),
    'Alpha': (8, 12),
    'Beta' : (12, 30),
    'Gamma': (30, 45),
}

nperseg = min(256, filtered.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# --- 왼쪽: 파워 스펙트럼 ---
for i in range(N_EEG):
    f, pxx = sp_signal.welch(filtered[i], fs=FS, nperseg=nperseg)
    axes[0].semilogy(f, pxx, linewidth=1.2, label=f'Ch{i+1}')

axes[0].set_xlabel('주파수 (Hz)')
axes[0].set_ylabel('파워 (µV²/Hz)')
axes[0].set_title('파워 스펙트럼', fontweight='bold')
axes[0].set_xlim(0, HIGH + 5)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, which='both')

# --- 오른쪽: 대역별 파워 막대 ---
band_table = {}
for i in range(N_EEG):
    f, pxx = sp_signal.welch(filtered[i], fs=FS, nperseg=nperseg)
    total  = np.trapezoid(pxx, f) if hasattr(np, 'trapezoid') else np.trapz(pxx, f)
    row = {}
    for name, (lo, hi) in BANDS.items():
        m = (f >= lo) & (f <= hi)
        p = np.trapezoid(pxx[m], f[m]) if hasattr(np, 'trapezoid') else np.trapz(pxx[m], f[m])
        row[name] = 100 * p / total if total > 0 else 0
    band_table[f'Ch{i+1}'] = row

band_df = pd.DataFrame(band_table).T[list(BANDS.keys())]
band_df.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_ylabel('전체 대비 비율 (%)')
axes[1].set_title('대역별 파워 비율', fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('대역별 파워 비율 (%)')
display(band_df.round(1))

---
# 13단계 · 저장

원본과 필터링된 신호를 **CSV**로 저장합니다.
Excel, MATLAB, 파이썬 어디서든 다시 열어 볼 수 있습니다.

저장 위치는 프로젝트의 `outputs/` 폴더입니다.

In [ ]:
out_dir = Path('..') / 'outputs'
out_dir.mkdir(exist_ok=True)

stamp = datetime.now().strftime('%Y%m%d_%H%M%S')

save_df = df.copy()
for i in range(N_EEG):
    save_df[f'Ch{i+1}_filtered'] = filtered[i]

csv_path = out_dir / f'ganglion_{stamp}.csv'
save_df.to_csv(csv_path, index=False)

print('저장 완료')
print(f'  파일  : {csv_path.resolve()}')
print(f'  크기  : {save_df.shape[0]} 행 x {save_df.shape[1]} 열')
print(f'  기간  : {save_df.shape[0] / FS:.2f} 초')
print()
print('열 구성:')
print(f'  time_sec           시간 (초)')
print(f'  Ch1..Ch{N_EEG}            원본 (µV)')
print(f'  Ch1_filtered..     필터 후 (µV)')
print()
print('나중에 다시 불러오기:')
print(f"  df = pd.read_csv(r'{csv_path.name}')")

---
# 14단계 · 연결 해제 ⚠️ **꼭 실행하세요**

보드와의 연결을 정리합니다.

이걸 건너뛰면 동글이 계속 점유된 상태로 남아서,
**다음 실행이나 OpenBCI GUI가 연결되지 않습니다.**

> 노트북 커널을 재시작해도 정리됩니다. 연결이 이상하면 커널 재시작 후 다시 시도하세요.

In [ ]:
try:
    if board.is_prepared():
        board.release_session()
        print('연결 해제 완료. 동글이 사용 가능한 상태입니다.')
    else:
        print('이미 해제되어 있습니다.')
except Exception as e:
    print(f'해제 중 문제 발생: {e}')
    print('노트북 커널을 재시작하면 정리됩니다.')

---
# 정리

## 오늘 배운 흐름

```
Ganglion 보드
     ↓  BLE
BLED112 동글 (COM 포트)
     ↓  BrainFlow
행렬 (채널 x 시간)
     ↓  필터
깨끗한 신호
     ↓
  ┌──────┬──────────┐
  RMS      주파수 분석
(세기)     (대역별 파워)
     ↓
   CSV 저장
```

## 핵심 개념

| 개념 | 의미 |
| --- | --- |
| 샘플링 레이트 | 초당 측정 횟수 (Ganglion = 200 Hz) |
| 나이퀴스트 | 볼 수 있는 최대 주파수 = 샘플링 레이트 ÷ 2 = 100 Hz |
| 대역통과 필터 | 원하는 주파수 구간만 남김 |
| 노치 필터 | 특정 주파수(60Hz 전원)만 제거 |
| RMS | 신호 세기 → 임계값 제어에 사용 |
| 파워 스펙트럼 | 주파수별 에너지 분포 |

---

## 과제 아이디어

**기초**
1. 눈 감고 20초 / 눈 뜨고 20초 → Alpha 파워 비교
2. 필터 대역을 바꿔 가며 신호가 어떻게 달라지는지 기록
3. 노치 필터를 껐을 때 스펙트럼의 60Hz 부근 관찰

**응용**
4. 팔 근육에 전극을 붙이고 힘줄 때 RMS 변화 측정 (EMG, `LOW=20, HIGH=95`)
5. RMS 임계값을 정해 *주먹 쥠 / 폄* 을 자동 판정하는 코드 작성
6. 판정 결과로 아두이노 서보 모터 제어 (MATLAB 실습의 파이썬 버전)

---

## 문제가 생기면

| 증상 | 확인할 것 |
| --- | --- |
| 연결 실패 | **OpenBCI GUI 종료** (가장 흔함) |
| 포트를 못 찾음 | 동글 다시 꽂기 → 2단계 재실행 |
| 샘플 0개 | 보드 전원, 배터리, 전극 연결 |
| 엉뚱한 보드에 연결됨 | `BOARD_MAC` 에 내 보드 MAC 지정 |
| 신호가 온통 잡음 | 전극 접촉, 레퍼런스 전극 위치 확인 |
| 계속 이상함 | 커널 재시작 → 처음부터 다시 |

---

*재활재생개론 · OpenBCI Ganglion 실습*